# Renewable Share and Merge

After importing the **global_power_plant_database** dataset, the renewable share index is computed and added as a column to the dataset.

The "renewable share index" defined as: **Renewable Share = Total renewable capacity / Total installed capacity**.

In [ ]:
import pandas as pd

# Carica dataset power plant
df = pd.read_csv("../Datasets_and_maps/global_power_plant_database.csv")

# Filtra solo impianti rinnovabili
renewables = ['Solar', 'Wind', 'Hydro', 'Geothermal', 'Biomass']

# Capacità totale da rinnovabili per paese
df_ren = df[df['primary_fuel'].isin(renewables)]
df_renewable_capacity = df_ren.groupby('country_long')['capacity_mw'].sum().reset_index()
df_renewable_capacity.columns = ['Country', 'Renewable_Capacity_MW']

# Capacità totale complessiva per paese
df_total_capacity = df.groupby('country_long')['capacity_mw'].sum().reset_index()
df_total_capacity.columns = ['Country', 'Total_Capacity_MW']

# Unisci le due tabelle (left join per mantenere tutti i paesi con impianti totali)
df_capacity = pd.merge(df_total_capacity, df_renewable_capacity, on='Country', how='left')

# Riempie eventuali NaN (paesi senza rinnovabili) con 0
df_capacity['Renewable_Capacity_MW'] = df_capacity['Renewable_Capacity_MW'].fillna(0)

# Aggiungi anche la quota rinnovabili (opzionale ma utile)
df_capacity['Renewable_Share'] = df_capacity['Renewable_Capacity_MW'] / df_capacity['Total_Capacity_MW']

C:\Users\2830d\AppData\Local\Temp\ipykernel_2308\618691665.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Datasets_and_maps/global_power_plant_database.csv")


The new dataset is composed of the following attributes:
+   **Country**
+   **Total_Capacity_MW**: computed as the sum of *capacity_mw* of all power plants in a country.
+   **Renewable_Capacity_MW**: computed as the sum of *capacity_mw* of all power plants that use renewable energy in a country.
+   **Renewable_Share**: as described before.

In [30]:

df_capacity

,Country,Total_Capacity_MW,Renewable_Capacity_MW,Renewable_Share
0,Afghanistan,300.550,258.550,0.860256
1,Albania,1529.000,1431.000,0.935906
2,Algeria,15873.800,694.800,0.043770
3,Angola,1071.180,770.600,0.719394
4,Antarctica,7.600,1.000,0.131579
...,...,...,...,...
162,Vietnam,41350.490,18576.490,0.449245
163,Western Sahara,23.400,0.000,0.000000
164,Yemen,1045.000,0.000,0.000000
165,Zambia,2689.337,2219.737,0.825384


### Merge

After reading the second dataset (**RISE_dataset**), the two are merged with a **inner join**.

In [31]:
df2 = pd.read_excel("../Datasets_and_maps/RISE_dataset.xlsx", header = 1)
df2


,Country,Region,Income group,Unnamed: 3,Energy Access score,Renewable Energy score,Energy Efficiency score,Overall score,Unnamed: 8,Indicator 1: Existence and monitoring of officially approved electrification plan,...,Indicator 11: Types of electricity rate structures,indicator 12: Carbon Pricing and Monitoring Mechanism,Unnamed: 30,Indicator 1: Legal framework for renewable energy,Indicator 2: Planning for renewable energy expansion,Indicator 3: Incentives and regulatory support for renewable energy,Indicator 4: Attributes of financial and regulatory incentives,Indicator 5: Network connection and pricing,Indicator 6: Counterparty risk,indicator 7: Carbon Pricing and Monitoring Mechanism
0,Afghanistan,South Asia,Low income,NaN,23.992691,26.934385,17.898479,22.941852,NaN,0,...,33.333333,0.0,NaN,50,67.708333,0.0,0.000000,61.111111,9.721250,0.0
1,Algeria,Middle East & North Africa,Upper middle income,NaN,100.000000,51.101032,55.987654,69.029562,NaN,100,...,68.518519,0.0,NaN,100,86.875000,87.5,55.555556,16.666667,11.110000,0.0
2,Angola,Sub-Saharan Africa,Upper middle income,NaN,48.298507,17.361071,17.749669,27.803083,NaN,80,...,50.000000,0.0,NaN,0,31.250000,37.5,0.000000,16.666667,36.110833,0.0
3,Argentina,Latin America & Caribbean,High income,NaN,100.000000,52.781026,43.946208,65.575745,NaN,100,...,53.703704,0.0,NaN,100,36.937500,75.0,55.555556,33.333333,68.640790,0.0
4,Armenia,Europe & Central Asia,Lower middle income,NaN,100.000000,63.071833,41.914683,68.328839,NaN,100,...,66.666667,0.0,NaN,100,60.312500,75.0,100.000000,33.333333,72.857000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,"Venezuela, RB",Latin America & Caribbean,High income,NaN,100.000000,24.704107,42.082782,55.595630,NaN,100,...,81.481481,0.0,NaN,100,42.375000,12.5,0.000000,0.000000,18.053750,0.0
107,Vietnam,East Asia & Pacific,Lower middle income,NaN,100.000000,64.092560,70.578704,78.223754,NaN,100,...,77.777778,0.0,NaN,100,60.156250,75.0,100.000000,66.666667,46.825000,0.0
108,"Yemen, Rep.",Middle East & North Africa,Lower middle income,NaN,19.235747,24.404623,12.114198,18.584856,NaN,0,...,37.000000,0.0,NaN,50,62.500000,37.5,11.111111,0.000000,9.721250,0.0
109,Zambia,Sub-Saharan Africa,Lower middle income,NaN,61.446662,46.655607,20.631614,42.911294,NaN,80,...,72.222222,0.0,NaN,100,18.750000,75.0,66.666667,16.666667,49.505914,0.0


Only usefull attributes are keeped, the other are discarded.

In [35]:
merged = pd.merge(df2, df_capacity, on = "Country")
merged_clean = merged[["Country", "Region", "Income group", "Renewable Energy score", "Total_Capacity_MW", "Renewable_Capacity_MW", "Renewable_Share"]]
merged_clean

,Country,Region,Income group,Renewable Energy score,Total_Capacity_MW,Renewable_Capacity_MW,Renewable_Share
0,Afghanistan,South Asia,Low income,26.934385,300.55000,258.55000,0.860256
1,Algeria,Middle East & North Africa,Upper middle income,51.101032,15873.80000,694.80000,0.043770
2,Angola,Sub-Saharan Africa,Upper middle income,17.361071,1071.18000,770.60000,0.719394
3,Argentina,Latin America & Caribbean,High income,52.781026,32913.07900,10727.09000,0.325922
4,Armenia,Europe & Central Asia,Lower middle income,63.071833,3271.00000,965.00000,0.295017
...,...,...,...,...,...,...,...
87,United Kingdom,OECD high income,High income,88.874008,97155.28448,39826.08448,0.409922
88,Uzbekistan,Europe & Central Asia,Lower middle income,30.059405,12640.00000,1240.00000,0.098101
89,Vietnam,East Asia & Pacific,Lower middle income,64.092560,41350.49000,18576.49000,0.449245
90,Zambia,Sub-Saharan Africa,Lower middle income,46.655607,2689.33700,2219.73700,0.825384


The merged and clean dataset is written in a **csv** file in the same folder of the other datasets.

In [37]:
merged_clean.to_csv("../Datasets_and_maps/merged.csv")